# Master notebook: LLM fine-tuning and alignment on a free T4 GPU

**Contents**
- The project: what it is, the components, the practices
- Setup: notebook settings, dependencies, machine check, save folder
- Data pipeline: look, clean, deduplicate, split, check, save
- Later sections are added when we reach them: SFT, ablations, DPO, reward model, PPO, evaluation, serving

---
# The project

## What this project is
We build a **technical Q&A assistant**: it answers technical questions the way a well-voted Stack Exchange answer does.

**Research question:** how far can a small open model be improved for one domain, using instruction tuning, parameter-efficient fine-tuning and preference alignment, on a single free 16 GB T4?

| Choice | Decision |
|---|---|
| Base model | Qwen/Qwen2.5-1.5B-Instruct (fallback: 0.5B if PPO is too slow or unstable) |
| Training method | Parameter-efficient only (LoRA, QLoRA, DoRA, IA3). No full fine-tuning: it does not fit a T4 |
| Data | Stack Exchange paired dataset: a question with a better-voted and a worse-voted answer |
| Hardware | Free T4 (Kaggle or Colab): 16 GB VRAM, no bf16 (so fp16), sessions end, limited disk |

## The components
| # | Component | Question it answers |
|---|---|---|
| Data | Clean, deduplicate, split | Can we trust the numbers we will report? |
| E0 | Baseline of the untrained model | What does the model do before we touch it? |
| E1 / E2 | Instruction tuning with LoRA / QLoRA | Can the model learn the answer style and domain? How much memory does QLoRA save? |
| E3 | Ablations: rank, data size, epochs, DoRA, IA3 | Which settings actually matter? |
| E4 | DPO (preference tuning) | Does learning better vs worse answers help beyond SFT? |
| E5 | Reward model | Can a small model score answer quality? |
| E6 | Miniature PPO / RLHF (custom loop) | What does RLHF look like when we build it ourselves? |
| E7 | DPO vs PPO | Which alignment method works better here? |
| E8 | Final evaluation and error analysis | Where does the final model still fail, and why? |
| Serving | FastAPI + Docker + Gradio demo | Can someone use the models? |

## How the pieces connect
```
raw Stack Exchange data
   -> clean + dedup + split by question ->  sft | pref | val | test
base model -> E0 baseline
           -> SFT (E1/E2/E3) -> DPO (E4) -> compare (E7)
           -> reward model (E5) -> PPO (E6) ->
all models -> same test set -> E8 evaluation -> demo
```

## Practices we follow
| Practice | Why |
|---|---|
| Test set is never used for training or tuning | Otherwise reported scores are fake (Lesson 2) |
| Split by question, not by row | One question has many rows; row splits leak |
| Fixed random seeds | Runs can be repeated and compared |
| One config (YAML) per experiment | Every result can be traced to its settings |
| Checkpoint to persistent storage (Kaggle output, Hugging Face Hub), resume after a disconnect | Sessions can end without warning |
| Every number comes from a real run output | We never invent results; missing numbers stay empty |
| Weights & Biases tracking, adapters and data versioned on the Hugging Face Hub | Experiments stay reproducible |
| Tests and linting on CPU in GitHub Actions; an evaluation gate | A new model that scores worse than the best one is rejected |
| Notebook first, then modules | Learn each step by seeing it work, then move logic into `src/` |

---
# Setup

## Notebook settings on Kaggle
In the notebook settings (right sidebar), set the **Accelerator** to a GPU (T4) and **Internet** to **On**. Internet needs a phone-verified account; without it the dataset cannot be streamed.

## Install dependencies
The data pipeline only needs `datasets`, `transformers` (for the Qwen tokenizer) and `numpy`. Kaggle and Colab usually have them already; this cell installs a package only if it is missing and never upgrades.
**Version pins are pending**: after the machine check below we pin exact versions that match the installed torch/CUDA.
Later parts will add `peft`, `trl`, `bitsandbytes`, `accelerate`, `wandb`.

In [ ]:
!pip install -q datasets transformers numpy

## Check the machine
Kaggle may show two T4s. Our design uses one, so the first lines list the GPUs and then pin the code to the first one. Paste the output back to the mentor.

In [ ]:
!nvidia-smi -L

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # use one T4, as designed

import platform
import torch, datasets, transformers

print("python", platform.python_version())
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus visible to torch:", torch.cuda.device_count())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("datasets", datasets.__version__)
print("transformers", transformers.__version__)

## Choose where to save files
Kaggle keeps `/kaggle/working` as the notebook's output when you save a version; files elsewhere are lost when the session ends. Later we also push checkpoints to the Hugging Face Hub for safety. Not needed for Step 1, but we set it up once now.

In [ ]:
import os
BASE = "/kaggle/working/llm-alignment-t4"
os.makedirs(BASE, exist_ok=True)
print("working folder:", BASE)

---
# Data pipeline
Planned steps (the list can change as we learn):
1. Look at the raw data
2. Find the site of each question
3. Count tokens in an answer
4. Filter by length
5. Filter out other bad rows (empty, identical, link-only)
6. Keep one pair per question
7. Keep only technical sites
8. Remove near-duplicate questions
9. Split into sft / pref / val / test by question
10. Check for leakage
11. Run on the full stream and save the files

## Step 1: Look at the raw data

### What
Read a handful of raw rows from the dataset and print them. No cleaning, no splitting.

### Why
You cannot clean data you have never seen. Every later step depends on knowing the exact columns and what they contain.

### How
- The dataset is huge, so we **stream** it: rows are fetched over the network one at a time and nothing big is saved on the notebook's disk.
- `take(5)` stops after 5 rows.
- Each row is a Python dictionary: `qid`, `question`, `date`, `metadata`, `response_j` (the better answer), `response_k` (the worse answer).

### Read 5 rows from the real dataset

In [ ]:
from datasets import load_dataset

stream = load_dataset(
    "lvwerra/stack-exchange-paired",
    data_dir="data/finetune",
    split="train",
    streaming=True,
)
rows = list(stream.take(5))

print("rows read:", len(rows))
print("columns:", list(rows[0].keys()))
print()
for r in rows:
    print(r["qid"], r["metadata"][1], "| chars: question", len(r["question"]),
          "better", len(r["response_j"]), "worse", len(r["response_k"]))

print("\n--- QUESTION ---\n", rows[0]["question"][:300])
print("\n--- BETTER ANSWER (response_j) ---\n", rows[0]["response_j"][:300])
print("\n--- WORSE ANSWER (response_k) ---\n", rows[0]["response_k"][:300])

---
## Step 2: Find the site of each question

### What
Turn the `metadata` field of a row into a clean site name such as `stackoverflow.com`.

### Why
The dataset mixes many Stack Exchange sites (Step 1 showed Stack Overflow, physics and travel). Our project is technical Q&A, so we need to know each row's site to keep the right ones. The site is also part of a question's identity: the same `qid` number can exist on two different sites.

### How
- `metadata` is a list of 3 strings: the question URL, the site URL, the author URL.
- The site is the second item, `metadata[1]`, for example `https://Stackoverflow.com`.
- `urlparse(...).netloc` keeps only the host part: `Stackoverflow.com`.
- `.lower()` makes it `stackoverflow.com`. Step 1 showed the dataset writes `Stackoverflow.com` with a capital S; without lowercasing, two spellings of one site would look like two sites.

### See the whole metadata list of one row

In [ ]:
for i, item in enumerate(rows[0]["metadata"]):
    print(i, item)

### Extract the site from every row

In [ ]:
from urllib.parse import urlparse

def get_site(metadata):
    return urlparse(metadata[1]).netloc.lower()

for r in rows:
    print(r["qid"], get_site(r["metadata"]))

---
## Step 3: Count tokens in an answer

### What
Measure how long a text is in **tokens**, using the Qwen tokenizer. We only measure here; the filter comes in Step 4.

### Why
The model never reads characters. It reads tokens. Our length limit (answers under about 200 tokens) and the GPU memory use are both counted in tokens. Characters are a poor guide: Step 1 showed an answer of 2171 characters, but how many tokens that is depends on the text, because code and prose split differently.

### How
- A **tokenizer** cuts text into pieces from a fixed vocabulary, and gives each piece a number (its id).
- The token count is the number of ids.
- We must use the tokenizer of the model we will train (Qwen2.5), because other models cut text differently.
- `add_special_tokens=False` counts only the text, without extra start or end markers.

### See how a sentence is split

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

print(tokenizer.tokenize("Use list[::-1] to reverse a list."))

### Count tokens for the better answer of each row

In [ ]:
def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

for r in rows:
    chars = len(r["response_j"])
    tokens = count_tokens(r["response_j"])
    print(r["qid"], "chars", chars, "tokens", tokens, "chars per token", round(chars / tokens, 2))

---
## Step 4: Filter by length

### What
Write a function that looks at one row and says why it is too long or too short, or says nothing if the row is fine.

### Why
- Our model trains on short answers (about 200 tokens or less), so long ones are removed.
- Preference training uses **both** answers of a pair, so both the better and the worse one must fit.
- Very short answers ("yes", "see docs") teach nothing, so we drop them too.
- Long questions use up GPU memory, so we cap the question length as well.
- All limits are in tokens, because Step 3 showed characters are a poor guide.

### How
- Three settings, written once at the top so they are easy to change: `MAX_ANSWER_TOKENS`, `MIN_ANSWER_TOKENS`, `MAX_QUESTION_TOKENS`.
- The function counts tokens with `count_tokens` from Step 3 and returns a short reason text, or `None` when the row passes.
- Returning the **reason** (not just yes or no) lets us count later how many rows each rule removed.

These limits are starting values. They come from our plan (short answers), not from a measurement, and we can change them.

In [ ]:
MAX_ANSWER_TOKENS = 200
MIN_ANSWER_TOKENS = 10
MAX_QUESTION_TOKENS = 256

def length_problem(row):
    q = count_tokens(row["question"])
    good = count_tokens(row["response_j"])
    bad = count_tokens(row["response_k"])
    if good > MAX_ANSWER_TOKENS or bad > MAX_ANSWER_TOKENS:
        return "answer too long"
    if good < MIN_ANSWER_TOKENS or bad < MIN_ANSWER_TOKENS:
        return "answer too short"
    if q > MAX_QUESTION_TOKENS:
        return "question too long"
    return None

### Try it on the 5 rows

In [ ]:
for r in rows:
    print(r["qid"],
          "question", count_tokens(r["question"]),
          "better", count_tokens(r["response_j"]),
          "worse", count_tokens(r["response_k"]),
          "->", length_problem(r) or "keep")

---
## Step 5: Filter out other bad rows

### What
Add a second function that catches rows with bad **content**: empty text, two identical answers, or an answer that is mostly a link. Then combine it with the length function from Step 4 into one function, `row_problem`.

### Why
- **Empty text** has nothing to learn from, and a missing field would crash later code.
- **Identical answers** give no better-versus-worse signal, so they are useless for preference training.
- **Mostly-a-link answers** ("see this page") contain no answer text. The model would learn to answer with links it cannot check.

### How
- `url_fraction` measures what share of an answer's characters belong to links. A regular expression finds the links; the function adds up their lengths and divides by the total length.
- An answer with a link inside real text is fine (a small fraction). We reject only when more than half is link (`MAX_URL_FRACTION = 0.5`). This is a judgment call, not a measured value.
- `content_problem` uses `or ""` so a missing value (`None`) is treated as empty and does not crash.
- `row_problem` runs the **cheap** content checks first and the token-counting length checks after, so we do not waste time tokenizing rows that are already bad.
- Like Step 4, it returns a reason (or `None`) so we can count removals per rule later.

In [ ]:
import re

URL_PATTERN = re.compile(r"https?://\S+")
MAX_URL_FRACTION = 0.5

def url_fraction(text):
    if not text:
        return 0.0
    return sum(len(u) for u in URL_PATTERN.findall(text)) / len(text)

def content_problem(row):
    q = (row["question"] or "").strip()
    good = (row["response_j"] or "").strip()
    bad = (row["response_k"] or "").strip()
    if not q or not good or not bad:
        return "empty text"
    if good == bad:
        return "identical answers"
    if url_fraction(good) > MAX_URL_FRACTION or url_fraction(bad) > MAX_URL_FRACTION:
        return "mostly a link"
    return None

def row_problem(row):
    return content_problem(row) or length_problem(row)

### Try it
The 5 real rows are probably all fine for these rules, so we also build a few fake rows that break each rule on purpose, to see the function work.

In [ ]:
ok = {
    "question": "How do I reverse a list in Python?",
    "response_j": "Use slicing: my_list[::-1] returns a reversed copy of the list.",
    "response_k": "Try the reversed function on the list and wrap it with list to get a new list back.",
}
fake_rows = {
    "good row": ok,
    "empty answer": dict(ok, response_k="   "),
    "missing answer": dict(ok, response_j=None),
    "identical answers": dict(ok, response_k=ok["response_j"]),
    "only a link": dict(ok, response_j="https://example.com/some/very/long/page/about/lists/and/reversing"),
    "link inside text": dict(ok, response_j="Slicing works well, see https://docs.python.org/3/ for more."),
}

print("--- fake rows ---")
for name, row in fake_rows.items():
    print(name.ljust(18), "->", row_problem(row) or "keep")

print("\n--- real rows ---")
for r in rows:
    print(r["qid"], "->", row_problem(r) or "keep")

---
## Step 6: Keep one pair per question

### What
Go through the rows and keep **only the first valid row of each question**. We store the kept rows in a dictionary called `pool`, and count how many rows each rule removed.

### Why
- One question appears in up to 10 rows, each with a different answer pair (Step 1 showed this: qid 1484683 appeared twice in just 5 rows).
- If we kept all of them, one popular question would dominate the training data, and it would make splitting by question harder to get right in Step 8.
- The cost is real: we throw away the other pairs of that question.

### How
- A question's identity is its **site plus its qid**, joined into one text key such as `stackoverflow.com|1484683`. The qid alone is not enough, because two sites can use the same number.
- `pool` is a Python dictionary: key to row. Asking `key in pool` is instant, even with thousands of rows.
- For each row, we check the cheap thing first (is the question already in the pool?), and only then run `row_problem`.
- "First valid row wins" means the survivor is not chosen for quality. The stream is shuffled, so it is effectively random.
- A row that fails `row_problem` is not added, so a later row of the same question can still get in.
- `Counter` is a dictionary that counts, which gives us the numbers to report later.

In [ ]:
from collections import Counter

def make_key(row):
    return f"{get_site(row['metadata'])}|{row['qid']}"

pool = {}              # key -> the row we keep
removed = Counter()    # reason -> number of rows removed

for r in rows:
    key = make_key(r)
    if key in pool:
        removed["question already kept"] += 1
        continue
    problem = row_problem(r)
    if problem:
        removed[problem] += 1
        continue
    pool[key] = r

print("rows seen:", len(rows))
print("questions kept:", len(pool))
print("removed:", dict(removed))
for key in pool:
    print(key)

---
## Step 7: Keep only technical sites

### What
Add a check that rejects any row whose site is not on an allowed list, and put the Step 6 loop into a reusable function, `build_pool`.

### Why
- Step 2 only *found* the site. Nothing removed rows from other sites yet, and Step 6's output still contained physics and travel questions.
- Our project is **technical** Q&A. Questions about physics or travel would teach the model a different domain and make the results harder to explain.
- The list is a decision, not a fact. It is a setting that can be changed.

### How
- `ALLOWED_SITES` is a Python **set**: checking `x in ALLOWED_SITES` is instant, however long the list is.
- Each name is written in lowercase, matching what `get_site` returns. If they differed in case, every row would be rejected without any error (the Step 2 lesson).
- The site check comes **first** in `row_problem`, because it is the cheapest: no token counting is needed to reject a physics question.
- `build_pool` is exactly the Step 6 loop wrapped in a function that returns the pool and the removal counts. We will call it again on the full stream.

In [ ]:
ALLOWED_SITES = {
    "stackoverflow.com",
    "superuser.com",
    "serverfault.com",
    "askubuntu.com",
    "unix.stackexchange.com",
    "softwareengineering.stackexchange.com",
    "dba.stackexchange.com",
    "datascience.stackexchange.com",
    "stats.stackexchange.com",
    "security.stackexchange.com",
}

def site_problem(row):
    if get_site(row["metadata"]) not in ALLOWED_SITES:
        return "site not allowed"
    return None

def row_problem(row):
    return site_problem(row) or content_problem(row) or length_problem(row)

### Run the loop again, now as a function

In [ ]:
from collections import Counter
def build_pool(rows):
    pool = {}
    removed = Counter()
    for r in rows:
        key = make_key(r)
        if key in pool:
            removed["question already kept"] += 1
            continue
        problem = row_problem(r)
        if problem:
            removed[problem] += 1
            continue
        pool[key] = r
    return pool, removed

pool, removed = build_pool(rows)
print("rows seen:", len(rows))
print("questions kept:", len(pool))
print("removed:", dict(removed))
for key in pool:
    print(key)

---
## Step 8: Remove near-duplicate questions

Different people post the same question in slightly different words. If one copy lands in training and the other in the test set, that is leakage, and splitting by `qid` cannot catch it because the two copies have different qids. We handle this in five small parts:

| Part | What it does |
|---|---|
| 8a | Cut a question into overlapping word groups (shingles) |
| 8b | Measure how similar two questions are |
| 8c | Shrink each question to a short fingerprint (MinHash) |
| 8d | Find candidate pairs without comparing every pair (LSH) |
| 8e | Plug it into `build_pool` |

### Step 8a: Cut a question into shingles

#### What
Turn a question into a **set of overlapping groups of 5 words**. Such a group is called a *shingle*, like roof shingles that overlap.

#### Why
- To say two questions are similar, we need to compare *parts* of them. Comparing the whole text only tells us "identical or not", and one changed word would make two almost-identical questions look different.
- Comparing single words is too loose: "dog bites man" and "man bites dog" use the same words but mean different things. Groups of consecutive words keep some of the word order.

#### How
- `text.lower()` ignores capital letters.
- `re.findall(r"\w+", ...)` pulls out the words: runs of letters, digits and underscores. Punctuation is dropped, so `don't` becomes `don` and `t`, and `list[::-1]` becomes `list` and `1`. That is rough, but the same rough rule is applied to every question, so comparisons stay fair.
- A window of `k` words slides along one word at a time. A question of N words gives N - k + 1 windows.
- The result is a Python **set**, so a group that appears twice counts once.
- A text shorter than `k` words becomes one single shingle, so it never crashes.

The first cell uses `k=3` on a short sentence so you can see every shingle. The real pipeline uses `k=5`.

In [ ]:
import re

def shingles(text, k=5):
    words = re.findall(r"\w+", text.lower())
    if len(words) <= k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i + k]) for i in range(len(words) - k + 1)}

example = "How do I reverse a list in Python"
print(re.findall(r"\w+", example.lower()))
print()
for piece in shingles(example, k=3):
    print(piece)

### Count the shingles of the real questions

In [ ]:
for r in rows:
    n_words = len(re.findall(r"\w+", r["question"]))
    n_shingles = len(shingles(r["question"]))
    print(r["qid"], "words", n_words, "shingles", n_shingles)

### Step 8b: Measure how similar two questions are

#### What
Turn "how similar are these two questions?" into **one number between 0 and 1**, using their shingle sets from 8a. The formula is called **Jaccard similarity**:

`similarity = (shingles in both) / (shingles in either)`

#### Why
- We need a number so we can set a rule such as "above 0.8 means near-duplicate".
- 0 means no shared shingles. 1 means the same set. A copy with one word changed still shares most shingles, so it scores high.

#### How
- `a & b` is the set of shingles found in both questions.
- `a | b` is the set of shingles found in at least one of them.
- Divide the size of the first by the size of the second. If both sets are empty we return 0, to avoid dividing by zero.

Tiny example with sets of 4 items: A = {1,2,3,4}, B = {3,4,5,6}. Both: {3,4} = 2 items. Either: {1,2,3,4,5,6} = 6 items. Similarity = 2/6 = 0.33.

In [ ]:
def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

print(jaccard({1, 2, 3, 4}, {3, 4, 5, 6}))
print(jaccard({1, 2, 3}, {1, 2, 3}))
print(jaccard({1, 2, 3}, {7, 8, 9}))

### Compare real questions

In [ ]:
sets = [shingles(r["question"]) for r in rows]

print("row 3 vs row 4 (same question twice):", jaccard(sets[3], sets[4]))
print("row 0 vs row 1 (different questions):", jaccard(sets[0], sets[1]))
print("row 0 vs row 3 (different questions):", jaccard(sets[0], sets[3]))

### Step 8c: Shrink each question to a short fingerprint (MinHash)

#### What
Replace each question's big shingle set (about 100 items, or thousands for long questions) with a **fingerprint: a list of 128 numbers**. The fraction of positions where two fingerprints agree **estimates the Jaccard similarity** of the two questions.

#### Why
- With 7,000 questions there are about 24 million pairs. Computing exact Jaccard on big sets for each is too slow, and on the full stream (many more questions) it is hopeless.
- Fingerprints all have the same small size, so comparing two of them is cheap, and they can be grouped quickly in 8d.

#### How
1. Turn each shingle into a number with a hash (`md5`, first 8 bytes).
2. Apply a "random shuffle" of those numbers: `(a * h + b) mod prime`, with fixed random `a` and `b`. We use 128 different shuffles, with a fixed seed so results are repeatable.
3. For each shuffle, keep only the **smallest** number across all shingles. That gives 128 numbers per question.
4. Why this works: for one shuffle, the smallest number belongs to a shingle chosen uniformly from the union of both questions. Both questions have the same minimum exactly when that chosen shingle is in the **intersection**. That happens with probability `|both| / |either|`, which is the Jaccard similarity. So the fraction of 128 positions that agree is an estimate of it.

The estimate is noisy with only 128 positions, so in 8e we will confirm candidates with the exact Jaccard from 8b.

In [ ]:
import hashlib
import random

NUM_PERM = 128
PRIME = (1 << 61) - 1
_rng = random.Random(0)
COEFS = [(_rng.randrange(1, PRIME), _rng.randrange(0, PRIME)) for _ in range(NUM_PERM)]

def shingle_hash(s):
    return int.from_bytes(hashlib.md5(s.encode()).digest()[:8], "big")

def minhash(shingle_set):
    hashes = [shingle_hash(s) for s in shingle_set]
    return [min((a * h + b) % PRIME for h in hashes) for a, b in COEFS]

def estimate_similarity(sig_a, sig_b):
    return sum(x == y for x, y in zip(sig_a, sig_b)) / len(sig_a)

sig = minhash(sets[0])
print("fingerprint length:", len(sig))
print("first 3 numbers:", sig[:3])

### Compare exact and estimated similarity

In [ ]:
sigs = [minhash(s) for s in sets]

pairs = [(3, 4), (0, 1), (0, 3)]
for i, j in pairs:
    print(f"row {i} vs row {j}: exact {jaccard(sets[i], sets[j]):.3f}  estimate {estimate_similarity(sigs[i], sigs[j]):.3f}")

### Test on a near-duplicate pair
The 5 real rows have no near-duplicate pair, so we make one: the same question with one word changed.

In [ ]:
base = ("how do i reverse a list in python without using the built in reverse function "
        "and keep the original order of the list intact for later use in my program")
variant = base.replace("intact", "unchanged")

a, b = shingles(base), shingles(variant)
print("exact   :", round(jaccard(a, b), 3))
print("estimate:", round(estimate_similarity(minhash(a), minhash(b)), 3))

### Step 8d: Find candidate pairs without comparing every pair (LSH)

#### What
Group the fingerprints from 8c so that **similar questions land in the same bucket**. Only questions sharing a bucket are compared. This trick is called **Locality-Sensitive Hashing (LSH)**, in its "banding" form.

#### Why
- Fingerprints are short, but comparing every pair is still about 24 million comparisons for 7,000 questions.
- Most pairs are obviously different. We want to skip them without looking.

#### How
1. Cut each 128-number fingerprint into **32 bands of 4 numbers**.
2. For each band, use `(band number, the 4 numbers)` as a dictionary key. Questions with the same key share a bucket.
3. Two questions become a **candidate pair** if they share a bucket in **at least one** band.

Why this favours similar questions: if two questions have similarity `s`, one band of 4 numbers matches with probability `s^4`. The chance that at least one of 32 bands matches is `1 - (1 - s^4)^32`. That is small for low `s` and close to 1 for high `s`.

Candidates are only suggestions. A few will not really be similar, so in 8e each candidate is checked with the exact Jaccard from 8b.

In [ ]:
BANDS = 32
ROWS = NUM_PERM // BANDS   # 4 numbers per band

print("chance that a pair becomes a candidate:")
for s in [0.2, 0.3, 0.5, 0.7, 0.8, 0.9]:
    print(f"  similarity {s}: {1 - (1 - s ** ROWS) ** BANDS:.3f}")

In [ ]:
from collections import defaultdict

def candidate_pairs(signatures):
    buckets = defaultdict(list)
    for idx, sig in enumerate(signatures):
        for b in range(BANDS):
            key = (b, tuple(sig[b * ROWS:(b + 1) * ROWS]))
            buckets[key].append(idx)
    pairs = set()
    for members in buckets.values():
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                pairs.add((members[i], members[j]))
    return pairs

In [ ]:
print("real rows:", sorted(candidate_pairs(sigs)))

extra = [base, variant]
extra_sigs = sigs + [minhash(shingles(t)) for t in extra]
print("with the base question and its variant added:", sorted(candidate_pairs(extra_sigs)))

### Step 8e: Plug near-duplicate removal into the pipeline

#### What
A function `remove_near_duplicates(pool)` that takes the pool of kept questions and drops every question that is a near-copy of an earlier one.

#### Why
This is the step that actually removes the leakage. 8a-8d were the tools.

#### How
1. Build the shingle set and fingerprint of every question in the pool.
2. Use `candidate_pairs` (8d) to get the few pairs worth checking.
3. For each candidate pair, compute the **exact** Jaccard (8b). If it is at least `NEAR_DUP_THRESHOLD`, drop the later question and keep the earlier one.
4. Return the cleaned pool and the number dropped.

Why this order: LSH is fast but approximate; exact Jaccard is slow but right. LSH narrows the pairs, and the exact check makes the final decision, so a false candidate is never removed.

Why this runs *after* `build_pool` and not inside it: near-duplicates need to see all questions, while `build_pool` handles one row at a time. Dedup on the pool is also cheaper because the pool is small.

The threshold is a setting. 0.8 is a common choice: LSH with 32 bands catches nearly every pair at or above it (8d table), and questions that similar are real copies. A lower threshold removes more, but starts removing questions that are merely related.

In [ ]:
NEAR_DUP_THRESHOLD = 0.8

def remove_near_duplicates(pool, threshold=NEAR_DUP_THRESHOLD):
    keys = list(pool)
    question_sets = [shingles(pool[k]["question"]) for k in keys]
    fingerprints = [minhash(s) for s in question_sets]

    dropped = set()
    for i, j in sorted(candidate_pairs(fingerprints)):
        if i in dropped or j in dropped:
            continue
        if jaccard(question_sets[i], question_sets[j]) >= threshold:
            dropped.add(j)

    kept = {k: pool[k] for idx, k in enumerate(keys) if idx not in dropped}
    return kept, len(dropped)

### Test on a small made-up pool

In [ ]:
other = ("what is the best way to center a div horizontally and vertically "
         "using css flexbox in a modern browser layout today")

toy_pool = {
    "a": {"question": base},
    "b": {"question": base + " thanks"},   # one word added
    "c": {"question": variant},            # one word changed
    "d": {"question": other},              # unrelated
}

print("a vs b:", round(jaccard(shingles(toy_pool["a"]["question"]), shingles(toy_pool["b"]["question"])), 3))
print("a vs c:", round(jaccard(shingles(toy_pool["a"]["question"]), shingles(toy_pool["c"]["question"])), 3))

for threshold in (0.8, 0.6):
    kept, n_dropped = remove_near_duplicates(toy_pool, threshold)
    print(f"threshold {threshold}: kept {list(kept)}, dropped {n_dropped}")

### Run the whole pool step on the 5 real rows

In [ ]:
pool, removed = build_pool(rows)
pool, n_near = remove_near_duplicates(pool)
if n_near:
    removed["near-duplicate question"] += n_near

print("questions kept:", len(pool))
print("removed:", dict(removed))

---
## Step 9: Split into sft / pref / val / test

#### What
Give every question in the pool to **exactly one** of four groups:

| Group | Used for | Target size |
|---|---|---|
| `sft` | instruction tuning (E1, E2, ablations) | about 5,000 |
| `pref` | preference training (DPO, reward model, PPO) | about 2,000 |
| `val` | choosing settings (beta, rank, epochs) | about 300 |
| `test` | the final score, never used for training or tuning | about 200 |

#### Why
- A question must never appear in two groups. If it did, the model would be tested on something it was trained on.
- Splitting by question, not by row, keeps all text of one question together. We already kept one pair per question (Step 6), and the near-duplicates are gone (Step 8). So similar questions cannot straddle two groups either. This is why the split comes *after* Step 8.
- The split must be the same on every run, even if the stream order changes or we re-run next week.

#### How
- Make a string from a seed and the question key: `"42:stackoverflow.com|1484683"`.
- Hash it with `md5` and turn the first 8 hex digits into a number between 0 and 1. Call it the question's **position**.
- Lay the groups along the 0-to-1 line, each as wide as its share (5000/7500 for `sft`, and so on). The group whose range holds the position is the question's group.
- The same key and the same seed always give the same group. It does not depend on the order of the rows or on which other questions exist.
- The shares follow from the target sizes, so counts come out approximately right, not exactly. In Step 11 we will look at the real counts.

In [ ]:
import hashlib

SPLIT_SEED = 42
SPLIT_FRACTIONS = {"sft": 5000 / 7500, "pref": 2000 / 7500, "val": 300 / 7500, "test": 200 / 7500}

def split_of(key, seed=SPLIT_SEED):
    digest = hashlib.md5(f"{seed}:{key}".encode()).hexdigest()
    position = int(digest[:8], 16) / 16 ** 8   # a number in [0, 1)
    cumulative = 0.0
    for name, fraction in SPLIT_FRACTIONS.items():
        cumulative += fraction
        if position < cumulative:
            return name
    return name   # guards against tiny rounding at the very end

print("fractions add up to:", sum(SPLIT_FRACTIONS.values()))

### Check that it is repeatable and does not depend on order

In [ ]:
key = "stackoverflow.com|1484683"
print("same key twice:", split_of(key), split_of(key))

for r in rows:
    print(r["qid"], split_of(make_key(r)))

### See the group sizes on 7,500 made-up keys
These keys are made up. They only show that the hash spreads questions in the right shares.

In [ ]:
from collections import Counter

fake_keys = [f"stackoverflow.com|{i}" for i in range(1, 7501)]
forward = Counter(split_of(k) for k in fake_keys)
backward = Counter(split_of(k) for k in reversed(fake_keys))

print(dict(forward))
print("same counts when the order is reversed:", forward == backward)